In [3]:
#任务1：取最近15天

import pandas as pd
import json
import chardet
from time import sleep
import time
import requests
import pandas as pd
import random
import pymysql
import json
import sys
import sqlalchemy
from datetime import datetime,date, timedelta
# with open(r"D:\WPS云盘\new\工具库\爬虫及大规模Pdf处理\土地数据爬取\土地市场网市县区code映射.json", 'rb') as f:
#     encoding = chardet.detect(f.read())['encoding']
# with open(r"D:\WPS云盘\new\工具库\爬虫及大规模Pdf处理\土地数据爬取\土地市场网市县区code映射.json", 'r', encoding=encoding) as f:
#     df_code = json.load(f)

# with open(r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\new\工具库\爬虫及大规模Pdf处理\土地数据爬取\土地市场网市县区code映射.json", 'rb') as f:
#     encoding = chardet.detect(f.read())['encoding']
# with open(r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\new\工具库\爬虫及大规模Pdf处理\土地数据爬取\土地市场网市县区code映射.json", 'r', encoding=encoding) as f:
#     df_code = json.load(f)
# df_code=df_code['data']
# results = []
# for df in df_code:
#     city_info = {
#             "省份": df['enumName'],
#             "省份代码": df['enumValue']
#         }
#     results.append(city_info)
# results_df = pd.DataFrame(results)


sql_engine = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'hz_work',
        'Hzinsights2015',
        'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
        '3306',
        'bond',
    ), poolclass=sqlalchemy.pool.NullPool
)
_cursor = sql_engine.connect()

def find_dgSj(json_obj, path=[]):
    try:
        if isinstance(json_obj, dict):
            for key, value in json_obj.items():
                if key == "cjJg":
                    return value
                else:
                    new_path = path + [key]
                    result = find_dgSj(value, new_path)
                    # print(key)
                    if result is not None:
                        return result
        elif isinstance(json_obj, list):
            for value in json_obj:
                result = find_dgSj(value, path)
                if result is not None:
                    return result
    except Exception as e:
        print(str(e))
        print(f"失败finddgsj")
    return None

def insert_err(code,pagenum):
    global proxies
    global run_count
    global start_time
    try:
        _db = pymysql.connect(host="rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com", user="qye",password="Hz123456", db="bond", charset='utf8')
        _cursor = _db.cursor()
        query = "INSERT IGNORE INTO err_landinfo (code,pagenum) VALUES (%s,%s)"
        data = (code,pagenum)
        _cursor.execute(query, data)
        _db.commit()
    except Exception as e:
        print(str(e))
        print(f"失败i1")
        insert_err(code,pagenum)

def insert_err1(gdGuid):
    global proxies
    global run_count
    global start_time
    try:
        _db = pymysql.connect(host="rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com", user="qye",password="Hz123456", db="bond", charset='utf8')
        _cursor = _db.cursor()
        query = "INSERT IGNORE INTO err_landinfo1 (gdGuid) VALUES (%s)"
        data = (gdGuid)
        _cursor.execute(query, data)
        _db.commit()
    except Exception as e:
        print(str(e))
        print(f"失败i2")
        insert_err1(gdGuid)

def insert_cjjg(gdGuid,df,cjjg):
    global proxies
    global run_count
    global start_time
    try:
        _db = pymysql.connect(host="rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com", user="qye",password="Hz123456", db="bond", charset='utf8')
        _cursor = _db.cursor()
        query = "UPDATE landinfo SET company = %s , cjjg= %s WHERE gdGuid = %s;"
        query1 = "UPDATE landinfo1 SET data = %s WHERE gdGuid = %s;"
        # query = """INSERT INTO landinfo_cjjg (data,company,cjjg,gdGuid) VALUES (%s,%s,%s,%s) 
        #         ON DUPLICATE KEY UPDATE
        #             data = VALUES(data),
        #             company = VALUES(company),
        #             cjjg = VALUES(cjjg);"""
        a=json.loads(df)
        if 'srDw' in a['relate'][0]:
            company=a['relate'][0]['srDw']
        elif 'srr' in a['data']:
            company=a['data']['srr']
        else:
            company=''
        data = [company,cjjg,gdGuid]
        data1 = [df]
        _cursor.execute(query, data)
        _cursor.execute(query1, data1)
        _db.commit() 
    except Exception as e:
        print(str(e))
        print(f"失败i3")
        insert_cjjg(gdGuid,df,cjjg)

def insert_whole(df):
    global proxies
    global run_count
    global start_time
    try:
        _db = pymysql.connect(host="rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com", user="qye",password="Hz123456", db="bond", charset='utf8')
        _cursor = _db.cursor()
        query = "INSERT IGNORE INTO landinfo_whole (data) VALUES (%s)"
        data = [df] 
        _cursor.execute(query, data)
        _db.commit() 
    except Exception as e:
        print(str(e))
        print(f"失败i4")
        insert_whole(df)
        
def insert_land(df,df_gd):
    global proxies
    global run_count
    global start_time
    df=df.where(pd.notna(df), None)
    try:
        # 连接到 MySQL 数据库
        _db = pymysql.connect(host="rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com", user="qye", password="Hz123456", db="bond", charset='utf8')
        _cursor = _db.cursor()

        # 定义插入语句
        query = "INSERT IGNORE INTO landinfo (gdGuid, xzqDm, tdZl, gyFs, gyMj, tdYt, qdRq, xzqFullName) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"
        query1 = "INSERT IGNORE INTO landinfo1 (gdGuid) VALUES (%s)"
        # 遍历 DataFrame 的每一行
        for index, row in df.iterrows():
            if row['gdGuid'] in df_gd:
                continue
            # 获取当前行的数据
            data = (row['gdGuid'], row['xzqDm'], row['tdZl'], row['gyFs'], row['gyMj'], row['tdYt'], row['qdRq'], row['xzqFullName'])
            data1= (row['gdGuid'])
            # 执行插入操作
            _cursor.execute(query, data)
            _cursor.execute(query1, data1)

        # 提交事务
        _db.commit()

    except Exception as e:
        print('失败i5')
        print(e)
        sleep(1)
        insert_land(df)
       

def get_content(bd,ed,num):
    global proxies
    global run_count
    global start_time
    url = "https://host.ratingdog.cn/api/services/app/Bond/QueryTradedHistoricalOfAgentTenants"
    
    headers_login = {
        '.Aspnetcore.Culture': 'zh-Hans',
        'authority': 'auth.ratingdog.cn',
        'ethod': 'POST',
        'path': '/api/TokenAuth/Authenticate',
        'cheme': 'https',
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate, br, zstd',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        'Content-Length': '100',
        'Content-Type': 'application/json;charset=UTF-8',
        'Devicechannel': 'gclife_bmp_pc',
        'Origin': 'https://www.ratingdog.cn',
        'Priority': 'u=1, i',
        'Ratingdog.Tenantid': '114',
        'Referer': 'https://www.ratingdog.cn/',
        'Sec-Ch-Ua': '"Not/A)Brand";v="8", "Chromium";v="126", "Microsoft Edge";v="126"',
        'Sec-Ch-Ua-Mobile': '?0',
        'Sec-Ch-Ua-Platform': '"Windows"',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site':'same-site',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 Edg/126.0.0.0'
    }

    # 定义 data
    data_login = {
        "UserNameOrEmailAddressOrPhone": "13918339361",
        "internationalPhoneCode": "86",
        "password": "welcome@1"
    }
    url_login='https://auth.ratingdog.cn/api/TokenAuth/Authenticate'
    r = requests.post(url_login, headers=headers_login, json=data_login)
    accessToken=r.json()['result']['accessToken']
    headers = {
        '.Aspnetcore.Culture': 'zh-Hans',
        'Host': 'host.ratingdog.cn',
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        "Authorization": f"Bearer {accessToken}",
        'Content-Type': 'application/json;charset=UTF-8',
        'Devicechannel': 'gclife_bmp_pc',
        'Origin': 'https://www.ratingdog.cn',
        'Ratingdog.Tenantid': '114',
        'Referer': 'https://www.ratingdog.cn/',
        'Sec-Ch-Ua': '"Chromium";v="122", "Not(A:Brand";v="24", "Microsoft Edge";v="122"',
        'Sec-Ch-Ua-Mobile': '?0',
        'Sec-Ch-Ua-Platform': '"Windows"',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-site',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36 Edg/122.0.0.0',
        }
    num=0
    totalCount=0
    while num<=totalCount:
        post_dict = {
        "BondTypes": [],
        "IssueMethods": [],
        "IssuerTypes": [],
        "YYIndustrys": [],
        "Citys": [],
        "YYRatings": [],
        "StartTradedDate": bd,
        "EndTradedDate":ed,
        "Natures": [],
        "MaxResultCount": 150,
        "SkipCount": num
        }
        def tryres():
            response = requests.post(url, headers=headers, json=post_dict, timeout=10)
            df=response.json()
            if df['success']!=True:
                df=tryres()
            return df
        
        df=tryres()
        df=pd.DataFrame(df['result']['items'])
        df.to_sql('cj',_cursor1,if_exists='append',index=False)
        num1=num/150
        print(f'成功{num1}')
        num+=150



get_content('2021-9-20','2022-3-17')
# for i in range(41,587):
#     get_content('2021-9-20','2022-3-17',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(208):
#     get_content('2024-5-17','2024-6-20',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(783):
#     get_content('2020-9-20','2021-3-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(435):
#     get_content('2020-6-20','2020-9-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(488):
#     get_content('2020-3-20','2020-6-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(624):
#     get_content('2019-9-20','2020-3-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(659):
#     get_content('2019-3-20','2019-9-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(763):
#     get_content('2018-3-20','2019-3-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(466):
#     get_content('2017-3-20','2018-3-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)
# for i in range(121):
#     get_content('2013-3-20','2017-3-19',i*150)
#     sleep_time = random.uniform(2, 6)
#     time.sleep(sleep_time)


成功0.0
成功1.0
成功2.0
成功3.0
成功4.0
成功5.0
成功6.0
成功7.0
成功8.0
成功9.0
成功10.0
成功11.0
成功12.0
成功13.0
成功14.0
成功15.0
成功16.0
成功17.0
成功18.0
成功19.0
成功20.0
成功21.0
成功22.0
成功23.0
成功24.0
成功25.0
成功26.0
成功27.0
成功28.0
成功29.0
成功30.0
成功31.0
成功32.0
成功33.0
成功34.0
成功35.0
成功36.0
成功37.0
成功38.0
成功39.0
成功40.0
成功41.0
成功42.0
成功43.0
成功44.0
成功45.0
成功46.0
成功47.0
成功48.0
成功49.0
成功50.0
成功51.0
成功52.0
成功53.0
成功54.0
成功55.0
成功56.0
成功57.0
成功58.0
成功59.0
成功60.0
成功61.0
成功62.0
成功63.0
成功64.0
成功65.0
成功66.0
成功67.0
成功68.0
成功69.0
成功70.0
成功71.0
成功72.0
成功73.0
成功74.0
成功75.0
成功76.0
成功77.0
成功78.0
成功79.0
成功80.0
成功81.0
成功82.0
成功83.0
成功84.0
成功85.0
成功86.0
成功87.0
成功88.0
成功89.0
成功90.0
成功91.0
成功92.0
成功93.0
成功94.0
成功95.0
成功96.0
成功97.0
成功98.0
成功99.0
成功100.0
成功101.0
成功102.0
成功103.0
成功104.0
成功105.0
成功106.0
成功107.0
成功108.0
成功109.0
成功110.0
成功111.0
成功112.0
成功113.0
成功114.0
成功115.0
成功116.0
成功117.0
成功118.0
成功119.0
成功120.0
成功121.0
成功122.0
成功123.0
成功124.0
成功125.0
成功126.0
成功127.0
成功128.0
成功129.0
成功130.0
成功131.0
成功132.0
成功133.0
成功134.0
成功135.0
成功136.0
成功137.0
成功138.

In [29]:
requests.post(url, headers=headers, json=post_dict, timeout=10)

NameError: name 'url' is not defined

In [2]:
import sqlalchemy
sql_engine1 = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'hz_work',
        'Hzinsights2015',
        'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
        '3306',
        'yq',
    ), poolclass=sqlalchemy.pool.NullPool
)
_cursor1 = sql_engine1.connect()